In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [ ]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [ ]:
type(sample_doc)

langchain_core.documents.base.Document

In [ ]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

/tmp/ipykernel_1506/451959863.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [ ]:
def load_all_pdfs():
    folder_path = "PDFS"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [ ]:
all_pdf_documents = load_all_pdfs()

total pdfs: 5
total pages: 10


In [ ]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [ ]:
chunks = split_docs(all_pdf_documents)

In [ ]:
len(chunks)

25

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


/tmp/ipykernel_1506/4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


In [ ]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [ ]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [ ]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (25, 384)
total documents added in vector store= 25
docs in collection: 25


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance / 2

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
results = rag_retriever.retrieve(
    "Where do I request access to a new tool?",
    top_k=10,
    score_threshold=-10   # disable filtering
)
for r in results:
    print(r["rank"], round(r["distance"], 3), r["metadata"]["source"].split("/")[-1], "|", r["document"][:80].replace("\n", " "))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 10 documents
1 0.905 05_Remote_Tools_and_Communication_Guide.pdf | Q: What if I'm going to miss a deadline? Flag it to your manager as early as pos
2 1.16 04_IT_and_Security_Policy.pdf | #security-incidents Slack channel or by emailing security@northlightanalytics.co
3 1.213 05_Remote_Tools_and_Communication_Guide.pdf | Remote Tools & Communication Guide Northlight Analytics | Internal Reference | U
4 1.278 05_Remote_Tools_and_Communication_Guide.pdf | Q: Where do I find past meeting recordings? All recorded meetings are automatica
5 1.433 02_New_Hire_Onboarding_Guide.pdf | Need Contact IT / hardware issues it-support@northlightanalytics.com Payroll & b
6 1.483 02_New_Hire_Onboarding_Guide.pdf | ● You will receive a welcome email with your company email address and temporary
7 1.511 02_New_Hire_Onboarding_Guide.pdf | New Hire Onboarding Guide Northlight Analytics | People Operations | For all new
8 1.527 04_IT_and_Security_Policy.pdf | IT & Security

In [ ]:
results = rag_retriever.retrieve(
    "What happens to my laptop and account when I leave the company?",
    top_k=5,
    score_threshold=-10
)
for r in results:
    print(r["rank"], round(r["distance"], 3), r["metadata"]["source"].split("/")[-1])
    print(r["document"])
    print("-----")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
1 0.839 04_IT_and_Security_Policy.pdf
Upon departure, all company devices must be returned within 10 business days using the prepaid shipping
label provided by IT. Account access is revoked on the employee's last working day at 11:59 PM in their
local time zone.
-----
2 1.185 04_IT_and_Security_Policy.pdf
2. Account Security
●
Multi-factor authentication (MFA) is mandatory on all company accounts, no exceptions.
●
Passwords are managed through the company 1Password vault; reused passwords are not permitted.
●
Access to production systems requires SSO login and is logged for audit purposes.
3. Network Requirements
Requirement
Detail
Home Wi-Fi
WPA2 or WPA3 encryption required, default router passwords must be changed
Public Wi-Fi
Company VPN required before accessing any internal tool
-----
3 1.331 04_IT_and_Security_Policy.pdf
VPN Provider
Company-managed VPN client, auto-installed via MDM
4. Data Handling
●
Customer data must never be s

In [ ]:
results = rag_retriever.retrieve(
    "How much is the home office stipend and when can I claim it?",
    top_k=5,
    score_threshold=-10
)
for r in results:
    print(r["rank"], round(r["distance"], 3), r["metadata"]["source"].split("/")[-1])
    print(r["document"])
    print("-----")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
1 0.999 01_Remote_Work_Policy.pdf
with their team's designated core hours, shown below.
Team
Core Hours (local team time)
Overlap Requirement
Engineering
10:00 AM - 2:00 PM
4 hours
Customer Success
9:00 AM - 1:00 PM
4 hours
Sales
8:00 AM - 12:00 PM
4 hours
People Ops
10:00 AM - 3:00 PM
3 hours
3. Home Office Stipend
Employees receive a one-time setup stipend and an annual refresh budget to maintain a safe and
productive home workspace.
●
One-time setup stipend: $750, available after 30 days of employment.
●
-----
2 1.097 03_Benefits_and_Compensation_Guide.pdf
$0
$140
Dental
$0
$25
Vision
$0
$12
Retirement
●
401(k) available immediately upon hire through Guideline.
●
Company match: 4% dollar-for-dollar, vesting immediately.
●
Contribution changes can be made at any time through the Guideline portal.
Paid Time Off
●
Flexible PTO policy - no fixed accrual cap, minimum 15 days per year strongly encouraged.
●
10 company-wide holidays, publish

In [ ]:
import re, glob, os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_folder = "/content/drive/MyDrive/RAG_CHATBOT/PDFS"
fallback = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150, separators=["\n\n", "\n", ". ", " "])

chunks = []  # list of (id, text, metadata)
for path in sorted(glob.glob(pdf_folder + "/*.pdf")):
    pages = PyMuPDFLoader(path).load()
    full_text = "\n".join(p.page_content for p in pages)   # keep this line
    full_text = re.sub(r"\nG\n", "\n- ", full_text)        # new line goes AFTER it
    name = os.path.basename(path)

    # split at numbered headings ("3. Home Office Stipend") and FAQ questions ("Q: ...")
    sections = re.split(
    r"\n(?=\d+\. [A-Z])|\n(?=Q: )|\n(?=(?:Before Day 1|Day 1 Checklist|Week 1 Goals|Week 2 Goals|Key Contacts|30-60-90 Day Check-ins)\n)",
    full_text
)
    i = 0
    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue
        parts = [sec] if len(sec) <= 800 else fallback.split_text(sec)
        for part in parts:
            chunks.append((f"{name}_{i}", part, {"source": name, "chunk_index": i}))
            i += 1

print("total chunks:", len(chunks))

total chunks: 34


In [ ]:
vector_store.collection = vector_store.client.get_or_create_collection(name="northlight_docs_v5")
print(vector_store.collection.count())   # should print 0

0


In [ ]:
ids   = [c[0] for c in chunks]
texts = [c[1] for c in chunks]
metas = [c[2] for c in chunks]

embs = embedding_manager.generate_embeddings(texts)

vector_store.collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=[e.tolist() for e in embs],
    metadatas=metas
)
print(len(chunks), vector_store.collection.count())   # both numbers must match

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

embeddings shape: (34, 384)
34 34


In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

queries = [
    "Where do I request access to a new tool?",
    "What happens to my laptop and account when I leave the company?",
    "How much is the home office stipend and when can I claim it?",
]
for q in queries:
    print("=" * 60)
    print("QUERY:", q)
    results = rag_retriever.retrieve(q, top_k=5, score_threshold=-10)
    for r in results:
        print(r["rank"], round(r["distance"], 3), r["metadata"]["source"], "|", r["document"][:70].replace("\n", " "))

QUERY: Where do I request access to a new tool?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
1 0.444 05_Remote_Tools_and_Communication_Guide.pdf | Q: How do I request access to a new tool? Post in #it-requests with th
2 1.051 04_IT_and_Security_Policy.pdf | 6. Approved Software Only software listed in the IT Asset Catalog may 
3 1.209 02_New_Hire_Onboarding_Guide.pdf | Key Contacts Need Contact IT / hardware issues it-support@northlightan
4 1.287 05_Remote_Tools_and_Communication_Guide.pdf | Remote Tools & Communication Guide Northlight Analytics | Internal Ref
5 1.308 05_Remote_Tools_and_Communication_Guide.pdf | Q: Who do I contact if a tool is down? Check #status-updates first for
QUERY: What happens to my laptop and account when I leave the company?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
1 0.985 04_IT_and_Security_Policy.pdf | 7. Offboarding Upon departure, all company devices must be returned wi
2 1.234 04_IT_and_Security_Policy.pdf | 2. Account Security - Multi-factor authentication (MFA) is mandatory o
3 1.245 04_IT_and_Security_Policy.pdf | 1. Device Requirements - Company-issued laptops must be used for all w
4 1.314 04_IT_and_Security_Policy.pdf | 4. Data Handling - Customer data must never be stored on local drives;
5 1.336 02_New_Hire_Onboarding_Guide.pdf | Day 1 Checklist Time (your local zone) Activity 9:00 AM Laptop setup +
QUERY: How much is the home office stipend and when can I claim it?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
1 0.672 01_Remote_Work_Policy.pdf | 3. Home Office Stipend Employees receive a one-time setup stipend and 
2 1.163 03_Benefits_and_Compensation_Guide.pdf | Benefits & Compensation Guide Northlight Analytics | People Operations
3 1.389 03_Benefits_and_Compensation_Guide.pdf | Paid Time Off - Flexible PTO policy - no fixed accrual cap, minimum 15
4 1.474 01_Remote_Work_Policy.pdf | Remote Work Policy Handbook Northlight Analytics | People Operations |
5 1.558 02_New_Hire_Onboarding_Guide.pdf | Before Day 1 - IT ships your laptop and peripherals to arrive 2-3 busi


In [ ]:
import os
from groq import Groq

client_llm = Groq(api_key=os.getenv("GROQ_API_KEY"))

key loaded


In [ ]:
SYSTEM_PROMPT = """You are an HR assistant for Northlight Analytics.
Answer ONLY using the provided context chunks.
Rules:
- Cite the source file name for each fact, like (05_Remote_Tools_and_Communication_Guide.pdf).
- If two sources give different answers to the same question, do NOT pick one. Start with "The documents disagree:" and list what each source says, with its file name.
- Include every relevant number, amount, and deadline from the context, not just the first one.
- If the context does not contain the answer, say "I don't have that information in the documents." Do not guess.
- If the context is ambiguous (for example, it is unclear whether a price is per person or in total), state the ambiguity and give the number exactly as written. Do NOT calculate or assume a total the documents don't state.
- Keep answers short and direct."""

def answer(question, top_k=5):
    results = rag_retriever.retrieve(question, top_k=top_k, score_threshold=-10)

    context = "\n\n".join(
        f"[Source: {r['metadata']['source']}]\n{r['document']}" for r in results
    )

    response = client_llm.chat.completions.create(
        model="openai/gpt-oss-120b",
        temperature=0,
        max_tokens=1000,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return response.choices[0].message.content, results

In [ ]:
for q in ["I'm on the PPO with a spouse and two kids, what's my monthly cost?"]:
    print("Q:", q)
    text, _ = answer(q)
    print("A:", text)

Q: I'm on the PPO with a spouse and two kids, what's my monthly cost?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents
A: Your plan’s listed costs are:

- Employee premium: **$0** per month  
- Dependent premium: **$180** per month (as shown in the “Dependent Cost (monthly)” column) (03_Benefits_and_Compensation_Guide.pdf)

The documents do not specify whether the $180 is per dependent or a total amount for all dependents. If it is per dependent, your spouse + 2 children would be 3 × $180 = **$540** per month; if it is a total amount for all dependents, the cost would be **$180** per month. The exact interpretation isn’t clarified in the provided materials.
